# Evaluation of Formulas from First Order Logic (Strict AST)

In this notebook, we show how formulas from *first order logic* can be evaluated in TypeScript, utilizing a purely array-based Abstract Syntax Tree (AST) strictly typed to prevent runtime errors.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from [group theory](https://en.wikipedia.org/wiki/Group_theory).
A group is defined as a triple $ \langle G, \mathrm{e}, \circ \rangle $
where:
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.

Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$
  * $\forall x: \exists y: y \circ x = \mathrm{e}$
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$

A group is *commutative* if, additionally, the following formula is satisfied:
  * $\forall x: \forall y: x \circ y = y \circ x$

We import our `recursive-set` types and the strict `ASTNode` definition from our new parser.

In [ ]:
import { LogicParser, ASTNode, UnaryNode, BinaryNode, 
         QuantifierNode, FunctionOrPredicateNode } from './FOL-Parser';

In [ ]:
import { Tuple, RecursiveMap as Map, RecursiveSet as Set, Value } from 'recursive-set';

Helper function to invoke the new parser and return a strictly typed AST.

In [ ]:
function parse(s: string): ASTNode {
    return new LogicParser(s).parse();
}

In [ ]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}
function map<K extends Value, V extends Value>(entries?: [any, V][]): Map<K, V> {
    const m = new Map<K, V>();    
    if (entries) {
        for (const [k, v] of entries) {
            // If the key is a standard array (like [0, 0]), convert it to a Tuple.
            // Otherwise, use it as-is (e.g., for string keys).
            const key = Array.isArray(k) ? tpl(...k) as unknown as K : k as K;
            m.set(key, v);
        }
    }
    return m;
}

Our new parser distinguishes between variables and function/predicate symbols as follows:
- Variables start with an **upper case** letter (e.g., `X`, `Y`).
- Functions and predicates start with a **lower case** letter (e.g., `e`, `p`).
- We can use arithmetic and relational operators natively.

Therefore, we represent the symbols from group theory as follows:
- The neutral element "$\mathrm{e}$" is the 0-ary function `e`.
- The operator "$\circ$" is represented by the native parser operator `*`.
- Equality "$=$" is natively supported by the parser as `=`.

Then the formulas of group theory can be represented as follows:

In [ ]:
const G1 = '∀X: e * X = X';
const G2 = '∀X: ∃Y: Y * X = e';
const G3 = '∀X: ∀Y: ∀Z: (X * Y) * Z = X * (Y * Z)';
const G4 = '∀X: ∀Y: X * Y = Y * X';

In [ ]:
const F1 = parse(G1);
console.dir(F1, { depth: null });

In [ ]:
const F2 = parse(G2);
console.dir(F2, { depth: null });

In [ ]:
const F3 = parse(G3);
console.dir(F3, { depth: null });

In [ ]:
const F4 = parse(G4);
console.dir(F4, { depth: null });

## A Structure for Group Theory

We use `recursive-set` types to map tuples of arguments to their function results or predicate truth values.

In [ ]:
type Universe                = Set<number>;
type FunctionInterpretation  = Map<Tuple<number[]>, number>;
type PredicateInterpretation = Set<Tuple<number[]>>;

interface Structure {
    universe:   Universe;
    functions:  Map<string, FunctionInterpretation>;
    predicates: Map<string, PredicateInterpretation>;
}

type VariableAssignment = Map<string, number>;

We define the universe `U` for a two-element group:

In [ ]:
const U: Universe = set(0, 1);

Define interpretations for `e` (0-ary) and `*` (binary).

In [ ]:
const NeutralElement: FunctionInterpretation = map([[[], 0]]);
NeutralElement

In [ ]:
const Product: FunctionInterpretation = map([[[0, 0], 0], [[0, 1], 1], [[1, 0], 1], [[1, 1], 0]])
Product

Define the interpretation for `=` (binary predicate).

In [ ]:
const Identity = U.map(x => tpl(x, x));

In [ ]:
const functions = map<string, FunctionInterpretation>();
functions.set('e', NeutralElement);
functions.set('*', Product);

In [ ]:
const predicates = map<string, PredicateInterpretation>();
predicates.set('=', Identity);

const S: Structure = { 
    universe:   U, 
    functions:  functions, 
    predicates: predicates 
};

Define a base variable assignment $\mathcal{I}$.

In [ ]:
const I: VariableAssignment = map<string, number>([['X', 0], ['Y', 1], ['Z', 0]]);

## Evaluation Functions

Because our AST is now strictly typed, we can separate the evaluation of mathematical Terms (which evaluate to a `number`) from Logical Formulas (which evaluate to a `boolean`).

In [ ]:
function evalTerm(t: ASTNode, S: Structure, I: VariableAssignment): number {
    // 1. Primitive Term (Variable or Number)
    if (typeof t === 'string') {
        if (/^[A-Z]/.test(t)) {
            const val = I.get(t);
            if (val === undefined) throw new Error(`Unbound variable: ${t}`);
            return val;
        }
        return parseFloat(t); 
    }
    
    // 2. Function Application (e.g. ['*', arg1, arg2] or ['e'])
    const [sym, ...args] = t;
    const argValues = args.map(arg => evalTerm(arg, S, I));
    const fJ = S.functions.get(sym);
    
    if (!fJ) throw new Error(`Function interpretation for '${sym}' not found`);
    return fJ.get(new Tuple(...argValues)) as number;
}

In [ ]:
function modify(I: VariableAssignment, variable: string, value: number): VariableAssignment {
    const Is = I.mutableCopy();
    Is.set(variable, value);
    return Is;
}

In [ ]:
function evalFormula(F: ASTNode, S: Structure, I: VariableAssignment): boolean {
    if (typeof F === 'string') {
        throw new Error(`Expected a logical formula, but got a mathematical term: ${F}`);
    }

    const op = F[0];

    switch (op) {
        // Constants
        case '⊤': return true;
        case '⊥': return false;

        // Unary Logic
        case '¬': {
            const [, body] = F as UnaryNode;
            return !evalFormula(body, S, I);
        }

        // Binary Logic
        case '∧':
        case '∨':
        case '→':
        case '↔': {
            const [, lhs, rhs] = F as BinaryNode;
            if (op === '∧') return evalFormula(lhs, S, I) && evalFormula(rhs, S, I);
            if (op === '∨') return evalFormula(lhs, S, I) || evalFormula(rhs, S, I);
            if (op === '→') return evalFormula(lhs, S, I) <= evalFormula(rhs, S, I);
            if (op === '↔') return evalFormula(lhs, S, I) === evalFormula(rhs, S, I);
            throw new Error("Unreachable");
        }

        // Quantifiers
        case '∀':
        case '∃': {
            const [, variable, body] = F as QuantifierNode;
            if (op === '∀') {
                return S.universe.every(c => evalFormula(body, S, modify(I, variable, c)));
            } else {
                return S.universe.some(c => evalFormula(body, S, modify(I, variable, c)));
            }
        }

        // If it isn't a logical connective, it is a Predicate (e.g. '=', '<', 'p')
        default: {
            const [sym, ...args] = F as FunctionOrPredicateNode;
            const argValues = args.map(arg => evalTerm(arg, S, I));
            const pJ = S.predicates.get(sym);
            
            if (!pJ) throw new Error(`Predicate interpretation for '${sym}' not found`);
            return pJ.has(new Tuple(...argValues));
        }
    }
}

## Checking whether $\mathcal{S}$ is a Group

In [ ]:
console.log(`evalFormula(G1) = ${evalFormula(F1, S, I)}`);
console.log(`evalFormula(G2) = ${evalFormula(F2, S, I)}`);
console.log(`evalFormula(G3) = ${evalFormula(F3, S, I)}`);
console.log(`evalFormula(G4) = ${evalFormula(F4, S, I)}`);

## Another Example

Let's show that the formula $\forall X: \exists Y: p(X,Y) \rightarrow \exists Y: \forall X: p(X,Y)$ is not universally valid.

In [ ]:
const F_invalid = parse('∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)');

In [ ]:
const U2: Universe = set(0, 1);

const pJ: PredicateInterpretation = set(tpl(0, 0), tpl(1, 1));
const predicates2 = map<string, PredicateInterpretation>();
predicates2.set('p', pJ);

const S2: Structure = { 
    universe: U2, 
    functions: map(), 
    predicates: predicates2 
};

const I2: VariableAssignment = map<string, number>();
I2.set('X', 0);
I2.set('Y', 0);

In [ ]:
console.log(`Result: ${evalFormula(F_invalid, S2, I2)}`);